# Snowflake

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/snowflake/snowflake.ipynb)

Snowflake is a fully managed, multi-cluster, shared-data cloud data platform: storage, compute, and services scale independently. This notebook walks from the architecture that makes that true through loading, transformation, governance, and cost, then finishes with a tuning chapter that turns the system functions and account-usage views into a step-by-step performance workflow.

The runnable cells use [DuckDB](https://duckdb.org) as a local stand-in for the engine: Parquet row groups play the role of micro-partitions, and Time Travel and streams are simulated so the mechanics are visible. Snowflake-only DDL stays in `sql` blocks.

*Last reviewed: November 2026, against the Snowflake documentation as of November 2026.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is Snowflake?

```sql
-- Confirm where you are connected and what compute exists.
SELECT CURRENT_ACCOUNT()  AS account,
       CURRENT_REGION()   AS region,
       CURRENT_VERSION()  AS version,
       CURRENT_ROLE()     AS role,
       CURRENT_WAREHOUSE() AS warehouse;

SHOW WAREHOUSES;

SELECT table_catalog, table_schema, COUNT(*) AS tables
FROM analytics.information_schema.tables
GROUP BY 1, 2
ORDER BY tables DESC;
```

Snowflake separates storage, compute, and services so each scales and is billed independently, and your job shifts from tuning servers to shaping queries and sizing warehouses.

The three layers are billed differently. Storage is charged per terabyte per month. A warehouse is charged in credits per second of uptime. Cloud services is charged only for the part of a day's cloud-services consumption that exceeds 10 percent of that day's warehouse compute credits, so ordinary parsing, optimization, and metadata work is effectively included. Serverless features sit outside that allowance: Snowpipe, automatic clustering, and materialized-view maintenance consume Snowflake-managed compute and are billed as serverless credits attributed to the feature that used them.

A concrete sense of scale: a MEDIUM warehouse running 8 hours a day for 22 working days consumes 4 x 8 x 22 = 704 credits. Running the same workload on XSMALL takes roughly four times as long but consumes nearly the same credits, because the credit rate and the runtime move in opposite directions. The two variables that actually change the bill are total warehouse uptime and how many bytes each statement reads.

The subtle trap is that the 10 percent cloud-services allowance is a daily, account-wide calculation. A day with very little warehouse compute makes the allowance tiny, so a metadata-heavy day can show billed cloud-services credits even though the workload never started a warehouse.

## How does Snowflake separate storage and compute?

```sql
-- Give each workload its own compute with its own lifecycle.
CREATE WAREHOUSE bi_wh        WAREHOUSE_SIZE = 'SMALL'  AUTO_SUSPEND = 300 AUTO_RESUME = TRUE;
CREATE WAREHOUSE elt_wh       WAREHOUSE_SIZE = 'LARGE'  AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;
CREATE WAREHOUSE adhoc_wh     WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;

-- Compute can be resized without touching data.
ALTER WAREHOUSE elt_wh SET WAREHOUSE_SIZE = 'XLARGE';
ALTER WAREHOUSE elt_wh SET WAREHOUSE_SIZE = 'LARGE';

-- Warehouse uptime and credit burn are visible per warehouse.
SELECT warehouse_name, warehouse_size, state, auto_suspend, auto_resume
FROM TABLE(information_schema.warehouse_load_history(
  date_range_start => DATEADD('hour', -3, CURRENT_TIMESTAMP())));
```

Storage and compute are decoupled services, so isolation and elasticity are configuration — but a bigger warehouse only helps when the work itself can be split.

The layers coordinate through cloud services rather than through each other. Storage holds many versions of each micro-partition, and the transaction manager records which version belongs to which committed transaction. A warehouse receives a plan that already names the partition versions to read, so two warehouses never need to agree on a cache state or a lock. A commit becomes visible to other warehouses as soon as the transaction manager publishes the new version.

```sql
-- Same query text, same unchanged tables, a different warehouse.
USE WAREHOUSE bi_wh;
SELECT COUNT(*) FROM analytics.orders;

USE WAREHOUSE adhoc_wh;
SELECT COUNT(*) FROM analytics.orders;   -- served by the account-wide result cache

-- A new commit invalidates that result and the next run reads storage again.
INSERT INTO analytics.orders (order_id, order_date, amount)
VALUES (999999, '2026-10-04', 10.00);
```

Isolation is a compute property, not a cache property. An identical query can be answered for a different warehouse from the shared result cache, so a two-warehouse benchmark can accidentally measure the cache instead of the new cluster; run the first query on each warehouse with different text or a changed table.

## What are virtual warehouses and how do they work?

```sql
CREATE OR REPLACE WAREHOUSE transform_wh
  WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';   -- heavy backfill
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'MEDIUM';  -- back to steady state

-- See current state and size for every warehouse in the account.
SHOW WAREHOUSES;

-- Suspend explicitly at the end of a batch window.
ALTER WAREHOUSE transform_wh SUSPEND;
```

A virtual warehouse is a metered cluster you start, size, and stop; isolate it by workload, never leave it idle, and remember that it changes speed, not the amount of data a query must read.

A warehouse is a pool of worker threads, not just a size. When more statements arrive than the pool can run at once, the extra statements wait in a queue, and `QUEUED_OVERLOAD_TIME` records that wait separately from `EXECUTION_TIME`. Queueing is a concurrency symptom, not a SQL symptom: the same query that completes in two seconds alone can sit in a queue for thirty seconds behind other work.

```text
XSMALL     1 credit/hour
SMALL      2
MEDIUM     4
LARGE      8
XLARGE    16
2XLARGE   32
3XLARGE   64
4XLARGE  128
5XLARGE  256
6XLARGE  512
```

In a multi-cluster warehouse the rate applies per cluster, so a warehouse allowed to reach six clusters can burn six times the listed rate. The size is only half the cost story; the cluster count and the suspend behavior are the other half.

## What is the cloud services layer?

```sql
SELECT query_id, total_elapsed_time / 1000 AS seconds,
       compilation_time / 1000 AS compile_s,
       credits_used_cloud_services,
       partitions_scanned, partitions_total,
       bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
ORDER BY total_elapsed_time DESC
LIMIT 20;

-- Serverless credits are attributed to the feature that consumed them.
SELECT service_type, SUM(credits_used) AS credits
FROM snowflake.account_usage.metering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY service_type
ORDER BY credits DESC;
```

Cloud services is Snowflake's always-on brain: it compiles queries, holds the metadata that enables pruning, and is free for ordinary work but metered for the serverless features it runs.

The 10 percent rule is the detail worth knowing precisely. Snowflake adds up the cloud-services credits and the warehouse compute credits for each day and bills cloud services only when they exceed 10 percent of that day's warehouse compute. The `ACCOUNT_USAGE` views show all consumed cloud-services credits, while `METERING_DAILY_HISTORY` shows what was actually billed, which is why the two numbers do not agree.

Cloud services also serves the result cache. A cache hit returns a stored result in milliseconds without starting a warehouse, and a cache miss is compiled into a plan and dispatched to compute. Because compilation and optimization happen here, a very complex query can spend noticeable time in `COMPILATION_TIME` before any warehouse work begins.

Do not add `CREDITS_USED_CLOUD_SERVICES` from `QUERY_HISTORY` straight onto the bill. Most of that consumption is absorbed by the 10 percent allowance, and only the daily excess is charged. Remember too that `ACCOUNT_USAGE` views can lag behind real time, so a query run a minute ago may not appear yet.

## What are micro-partitions and how does pruning work?

```sql
-- Ordered load: rows arrive sorted, so partition ranges are narrow.
INSERT INTO analytics.orders (order_id, order_date, customer_id, amount)
SELECT order_id, order_date, customer_id, amount
FROM staging.orders_raw
ORDER BY order_date, customer_id;

-- Good predicate: a literal range prunes partitions.
SELECT COUNT(*) FROM analytics.orders
WHERE order_date BETWEEN '2026-09-01' AND '2026-09-30';

-- Bad predicate: the cast hides the literal from partition pruning.
SELECT COUNT(*) FROM analytics.orders
WHERE CAST(order_date AS STRING) LIKE '2026-09%';

-- Measure pruning after the fact: scanned versus total partitions.
SELECT ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       partitions_scanned, partitions_total
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE '%analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

Micro-partitions plus per-column min/max statistics are the whole performance model: order rows by your common filter and pruning does the rest, while functions and type mismatches on the filter column throw it away.

A partition is closed when it reaches the target size, so the order of an insert becomes the physical order of the table. Snowflake does not reshuffle rows to fit a boundary and never rewrites a partition in place; an update or delete records the new row image in a new partition and marks the old one superseded. Statistics are recorded per column per partition, which is what makes a predicate on any single column eligible for pruning.

```text
unordered load                     ordered load
P1  2026-01-01 .. 2026-12-31       P1  2026-01-01 .. 2026-01-07
P2  2026-01-01 .. 2026-12-31       P2  2026-01-08 .. 2026-01-14
P3  2026-01-01 .. 2026-12-31       P3  2026-01-15 .. 2026-01-21
P4  2026-01-01 .. 2026-12-31       P4  2026-01-22 .. 2026-01-28

one-day filter -> 4 partitions      one-day filter -> 1 partition
```

An update breaks the ordering that a sorted load created. The rewritten rows land in new partitions that may span the whole date range again, so a table with continuous updates drifts back toward full scans unless it is reclustered or its load order is restored.

## What is a clustering key and when should you use one?

```sql
-- Declare the key: date first for range filters, then a high-cardinality id.
ALTER TABLE analytics.events CLUSTER BY (event_date, customer_id);

-- Depth: lower is better, and around 1 means near-perfect ordering.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events', '(event_date, customer_id)');

-- Depth without passing the key echoes the table's current clustering.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events');

-- Pause maintenance on a table that has stopped changing.
ALTER TABLE analytics.events SUSPEND RECLUSTERING;
-- Resume, or remove the key entirely.
ALTER TABLE analytics.events RESUME RECLUSTERING;
ALTER TABLE analytics.events DROP CLUSTERING KEY;

-- Serverless credit cost of automatic clustering, attributed per table.
SELECT table_name, SUM(credits_used) AS clustering_credits
FROM snowflake.account_usage.automatic_clustering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY table_name
ORDER BY clustering_credits DESC;
```

Cluster only large, repeatedly filtered tables, lead the key with the column used in range filters, and judge it by clustering depth and query improvement against the serverless credits it consumes.

Automatic clustering compares the table's current depth against the requested key and rewrites overlapping partitions until the order is restored. It does not run on a fixed schedule; Snowflake triggers it as disorder accumulates, so cost arrives in bursts after large loads. `SYSTEM$CLUSTERING_INFORMATION` reports `average_depth` and `average_overlaps`, where depth is the average number of partitions that must be probed for a point in the key space.

```text
key (order_date, customer_id)
before reclustering:  average_depth = 3.4   -> a point lookup probes about 3 partitions
after reclustering:   average_depth = 1.2   -> about 1 partition
```

A clustering key only helps predicates on its leading columns. If a workload filters on `customer_id` alone, a key of `(order_date, customer_id)` may not prune much, because the date ranges of neighboring partitions still overlap.

In [ ]:
import datetime as dt
import pathlib
import tempfile

import duckdb

con = duckdb.connect()
tmp = pathlib.Path(tempfile.mkdtemp())

# The same rows loaded in arrival order (random dates) and sorted by event_date; each row group ~ a micro-partition.
source = """
    SELECT i AS event_id, DATE '2026-01-01' + CAST((i * 7919) % 300 AS INTEGER) AS event_date, i % 50 AS amount
    FROM range(300000) t(i)
"""
con.execute(f"COPY ({source}) TO '{tmp}/arrival.parquet' (FORMAT parquet, ROW_GROUP_SIZE 4096)")
con.execute(f"COPY ({source} ORDER BY event_date) TO '{tmp}/sorted.parquet' (FORMAT parquet, ROW_GROUP_SIZE 4096)")


def profile(path: str) -> tuple[int, int, float]:
    """partitions scanned for one week, total partitions, and average clustering depth."""
    ranges = con.execute(f"""
        SELECT CAST(stats_min_value AS DATE), CAST(stats_max_value AS DATE)
        FROM parquet_metadata('{path}') WHERE path_in_schema = 'event_date'
    """).fetchall()
    scanned = sum(1 for lo, hi in ranges if hi >= DATE_LO and lo <= DATE_HI)
    # Depth: how many partitions overlap each partition's range (SYSTEM$CLUSTERING_DEPTH's idea).
    depth = sum(sum(1 for lo2, hi2 in ranges if lo2 <= hi and hi2 >= lo) for lo, hi in ranges) / len(ranges)
    return scanned, len(ranges), depth


DATE_LO, DATE_HI = dt.date(2026, 6, 1), dt.date(2026, 6, 7)
for name in ("arrival", "sorted"):
    scanned, total, depth = profile(f"{tmp}/{name}.parquet")
    print(f"{name:>7}: partitions scanned {scanned}/{total}, average depth {depth:.1f}")
arrival, sorted_ = profile(f"{tmp}/arrival.parquet"), profile(f"{tmp}/sorted.parquet")
assert arrival[0] == arrival[1] and sorted_[0] <= 3 and sorted_[2] <= 3

## How do you read a query profile and tune a slow query?

```sql
-- 1. Find the worst offenders in the last day.
SELECT query_id, query_type, warehouse_name, warehouse_size, user_name,
       total_elapsed_time / 1000 AS elapsed_s,
       queued_overload_time / 1000 AS queued_s,
       compilation_time / 1000 AS compile_s,
       execution_time / 1000 AS exec_s,
       partitions_scanned, partitions_total,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       percentage_scanned_from_cache,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       credits_used_cloud_services
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
ORDER BY total_elapsed_time DESC
LIMIT 25;

-- 2. Inspect one query in detail, including its text.
SELECT query_id, query_text, total_elapsed_time, execution_time,
       partitions_scanned, partitions_total,
       bytes_scanned, bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';

-- 3. Look at the plan, not just the timings.
SELECT SYSTEM$EXPLAIN_PLAN_JSON('<query_id>');
```

A query profile tells you which operator dominated and whether the cause was pruning, spilling, queuing, or query shape — read the numbers from `QUERY_HISTORY` first, then confirm in the operator tree before changing anything.

The profile is a tree, and a slow child makes every ancestor look slow, because a parent's elapsed time includes the child's. Read it bottom-up: find the operator with the largest share of execution time, fix that node's input, then re-read. The operator with the most rows is not necessarily the one to tune.

Compare the numbers in order rather than all at once: `QUEUED_OVERLOAD_TIME` versus `EXECUTION_TIME` (concurrency or SQL), `PARTITIONS_SCANNED` divided by `PARTITIONS_TOTAL` (pruning), `PERCENTAGE_SCANNED_FROM_CACHE` (warm or cold), and the two spill columns (memory). Only when queuing is negligible and the scan is already small is the remaining time a pure operator-cost problem.

`TOTAL_ELAPSED_TIME` includes compilation and queuing, so a query can look slow on a dashboard while the SQL itself is fast. Subtract the pieces before rewriting anything.

## How do you size a virtual warehouse?

```sql
-- Start small, work up only when a measured query is latency-bound.
CREATE OR REPLACE WAREHOUSE bi_wh WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 300 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;

ALTER WAREHOUSE bi_wh SET WAREHOUSE_SIZE = 'MEDIUM';   -- if p95 latency is unacceptable
ALTER WAREHOUSE bi_wh SET WAREHOUSE_SIZE = 'SMALL';    -- and back down after measuring

-- Credits and runtime per warehouse over the last week.
SELECT warehouse_name,
       SUM(credits_used) AS credits,
       SUM(credits_used_compute) AS compute_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;

-- Average and p95 query latency per warehouse.
SELECT warehouse_name,
       ROUND(AVG(execution_time) / 1000, 1) AS avg_exec_s,
       ROUND(APPROX_PERCENTILE(execution_time, 0.95) / 1000, 1) AS p95_exec_s,
       COUNT(*) AS queries
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY p95_exec_s DESC;
```

Size up for latency and scale out for concurrency, but expect roughly constant credits per unit of parallel work — so fix pruning and query shape before paying for a bigger warehouse.

Scale-up only accelerates the parts of a plan that can be spread across nodes. A skewed join key, a single-threaded aggregation, or one very wide row concentrates work and leaves extra nodes idle. Before resizing, check whether the dominant operator is parallel or serial; if it is serial, a bigger warehouse adds cost without adding speed.

A query that runs 600 seconds on MEDIUM (4 credits/hour) consumes 4 x 600 / 3600 = 0.667 credits. Run in 300 seconds on LARGE (8 credits/hour), the same query consumes 8 x 300 / 3600 = 0.667 credits. Scaling up is credit-neutral when the runtime halves; it is a loss when the runtime falls by less than the rate rises.

A resize applies to statements that start after it completes; a query already running keeps the size and resource pool it started with. Resize between batches, not in the middle of one.

## What are multi-cluster warehouses and when do you use them?

```sql
ALTER WAREHOUSE bi_wh SET
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 6
  SCALING_POLICY = 'STANDARD';

ALTER WAREHOUSE reporting_wh SET
  MIN_CLUSTER_COUNT = 2
  MAX_CLUSTER_COUNT = 10
  SCALING_POLICY = 'ECONOMY';

-- Queueing is the signal that scale-out is worth trying.
SELECT warehouse_name,
       COUNT(*) AS queries,
       SUM(CASE WHEN queued_overload_time > 0 THEN 1 ELSE 0 END) AS queued_queries,
       ROUND(SUM(queued_overload_time) / 1000, 1) AS total_queued_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY total_queued_s DESC;

-- Watch cluster count and queueing over time for one warehouse.
SELECT start_time, warehouse_name, avg_running, avg_queued_load
FROM TABLE(information_schema.warehouse_load_history(
  warehouse_name => 'BI_WH',
  date_range_start => DATEADD('hour', -12, CURRENT_TIMESTAMP())));
```

Multi-cluster warehouses absorb concurrency by adding clusters, so reach for them when queue time — not per-query scan cost — is the bottleneck, and pick `STANDARD` for interactive work and `ECONOMY` for batch.

Each cluster is a full copy of the warehouse and bills at the same per-hour rate, so `MAX_CLUSTER_COUNT` is a cost multiplier as well as a capacity setting. Clusters are added when queries are waiting and removed when they are not, subject to `MIN_CLUSTER_COUNT`. `MAX_CONCURRENCY_LEVEL` is the number of concurrent queries a single cluster is expected to absorb; a lower value makes Snowflake add the next cluster sooner.

STANDARD is reactive: it adds a cluster as soon as queries queue and removes clusters when load drops, which keeps interactive latency low. ECONOMY is deliberate: it waits for evidence that an extra cluster will be used for a while before adding it, so a burst may queue slightly longer but the warehouse avoids starting and stopping clusters repeatedly.

Adding clusters does not make a single query faster; every query still runs inside one cluster. If one query is slow with no queueing, multi-cluster is the wrong lever.

## How do auto-suspend and auto-resume affect cost?

```sql
ALTER WAREHOUSE bi_wh        SET AUTO_SUSPEND = 600 AUTO_RESUME = TRUE;  -- interactive bursts
ALTER WAREHOUSE elt_wh       SET AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;  -- scheduled batch
ALTER WAREHOUSE reporting_wh SET AUTO_SUSPEND = 0;                        -- costs idle credits

-- When did each warehouse start, suspend, or resize?
SELECT warehouse_name, previous_state, state, timestamp, user_name
FROM snowflake.account_usage.warehouse_events_history
WHERE timestamp >= DATEADD('day', -2, CURRENT_TIMESTAMP())
ORDER BY timestamp DESC
LIMIT 50;

-- Idle time is visible as long running stretches with no queries.
SELECT warehouse_name,
       SUM(credits_used) AS credits,
       COUNT(DISTINCT DATE_TRUNC('hour', start_time)) AS active_hours
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -3, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;
```

Auto-suspend is free money as long as you match the value to the workload: 60 seconds for batch, several minutes for interactive, never 0 — and keep auto-resume on so scheduled work cannot fail silently.

Auto-suspend is measured in seconds of warehouse inactivity, and the 60-second billing minimum applies to each start, so a warehouse that wakes for a five-second statement is billed for a minute. A multi-cluster warehouse pays that minimum per cluster, and resuming a warehouse also means its local SSD cache starts over.

A SMALL warehouse (2 credits/hour) left running through a 14-hour overnight gap consumes 2 x 14 = 28 idle credits. With `AUTO_SUSPEND = 60` it consumes roughly nothing after the last statement, at the cost of a cold cache on the next run.

Values below 60 seconds are effectively the 60-second minimum for a standard warehouse, so there is no benefit in setting an aggressive number. What matters is matching the value to how likely the next query is.

## What caching layers does Snowflake have?

```sql
-- How much of each query was served from cache, and were partitions even scanned?
SELECT query_id, warehouse_name, percentage_scanned_from_cache,
       partitions_scanned, partitions_total, bytes_scanned
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -3, CURRENT_TIMESTAMP())
ORDER BY start_time DESC
LIMIT 20;

-- Non-deterministic: never result-cached.
SELECT CURRENT_TIMESTAMP(), COUNT(*) FROM analytics.orders;

-- Deterministic and identical each time: eligible for the result cache.
SELECT COUNT(*) FROM analytics.orders;

-- Compare cache-aware timings across two runs of the same text.
SELECT query_id, total_elapsed_time / 1000 AS elapsed_s, percentage_scanned_from_cache
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE 'SELECT COUNT(*) FROM analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

Know the three caches — result cache (24 hours, free, invalidated by DML), warehouse SSD cache (lost on suspend), and the shared remote metadata cache (persistent) — and check `PERCENTAGE_SCANNED_FROM_CACHE` before assuming a query is cacheable.

The result cache is keyed by query text, the role and its privileges, the relevant session parameters, and the versions of the tables involved. It lives in cloud services and needs no warehouse, which is why a hit can return while the warehouse is suspended. The warehouse cache is raw micro-partition bytes on the nodes' local SSD and belongs to one warehouse. The metadata cache is shared account-wide and survives suspension.

```sql
SELECT COUNT(*) FROM analytics.orders;      -- runs, result stored
SELECT COUNT(*) FROM analytics.orders;      -- identical text, unchanged tables: cache hit
INSERT INTO analytics.orders (order_id) VALUES (999999);  -- new table version
SELECT COUNT(*) FROM analytics.orders;      -- re-executed even though the text is identical
```

A cache hit is invisible in `PARTITIONS_SCANNED` because nothing is scanned, so a `PERCENTAGE_SCANNED_FROM_CACHE` of zero can still mean the result cache answered the query. Check `TOTAL_ELAPSED_TIME` and whether a warehouse started before concluding the cache missed.

## What is Time Travel and Fail-safe?

```sql
ALTER TABLE analytics.orders SET DATA_RETENTION_TIME_IN_DAYS = 30;

-- Query data as of an absolute time, an offset, or after a specific statement.
SELECT COUNT(*) FROM analytics.orders AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);
SELECT COUNT(*) FROM analytics.orders AT (OFFSET => -3600);
SELECT COUNT(*) FROM analytics.orders BEFORE (STATEMENT => '<query_id>');

-- Recover a dropped object or snapshot a table without copying it.
UNDROP TABLE analytics.orders;
CREATE TABLE analytics.orders_audit CLONE analytics.orders AT (OFFSET => -86400);

-- How much history a table keeps, and how much storage old versions consume.
SELECT table_name, retention_time, dropped, created
FROM analytics.information_schema.tables
WHERE table_name = 'ORDERS';
```

Time Travel is a queryable safety net bounded by retention, Fail-safe is a seven-day support-only lifeline, and neither replaces replication and exports for real disaster recovery.

`AT` and `BEFORE` accept a timestamp, an offset in seconds, or a statement id. `BEFORE (STATEMENT => id)` returns the state just before that statement ran and excludes its effects; `AT (STATEMENT => id)` includes them. `UNDROP` restores a dropped object while its retention window is open, and a clone taken `AT` a timestamp is a durable way to keep a point-in-time copy after retention passes.

Retention is configurable at database, schema, and table level. Standard Edition allows 0 or 1 day; Enterprise Edition allows 0 to 90 days. Fail-safe adds a fixed seven days after Time Travel ends, applies to permanent tables only, and is not queryable: recovery requires Snowflake support. Transient and temporary tables get at most one day of Time Travel and no Fail-safe at all.

Longer retention is not free. Superseded micro-partitions remain stored for the whole window, so a heavily changing table with 90-day retention carries a storage bill proportional to that history. Transient tables are the cheaper choice for rebuildable staging where Fail-safe adds no value.

In [ ]:
# Every DML statement produces a new table version; old micro-partitions are retained, not overwritten.
versions: list[tuple[str, str]] = []  # (statement_id, snapshot table name)


def run_dml(statement_id: str, sql: str) -> None:
    con.execute(sql)
    snapshot = f"orders_v{len(versions)}"
    con.execute(f"CREATE TABLE {snapshot} AS SELECT * FROM orders")  # a real engine shares the partitions
    versions.append((statement_id, snapshot))


con.execute("CREATE TABLE orders (order_id INTEGER, status VARCHAR)")
run_dml("q1", "INSERT INTO orders VALUES (1, 'created'), (2, 'created'), (3, 'shipped')")
run_dml("q2", "UPDATE orders SET status = 'shipped' WHERE order_id = 1")
run_dml("q3", "DELETE FROM orders")  # the accident


def before(statement_id: str) -> str:
    """SELECT ... FROM orders BEFORE (STATEMENT => statement_id)"""
    position = [s for s, _ in versions].index(statement_id)
    return versions[position - 1][1]


print("now:", con.execute("SELECT count(*) FROM orders").fetchone()[0], "rows")
# Restore: CREATE TABLE orders_restored CLONE orders BEFORE (STATEMENT => 'q3')
con.execute(f"CREATE TABLE orders_restored AS SELECT * FROM {before('q3')}")
restored = con.execute("SELECT * FROM orders_restored ORDER BY order_id").fetchall()
print("restored:", restored)
assert restored == [(1, "shipped"), (2, "created"), (3, "shipped")]

## What is zero-copy cloning?

```sql
-- Clone an entire database for a safe development sandbox.
CREATE DATABASE analytics_dev CLONE analytics;

-- Snapshot one table at a point in time.
CREATE TABLE analytics.orders_snapshot CLONE analytics.orders
  AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);

-- The clone is independent as soon as its partitions are modified.
UPDATE analytics_dev.marts.daily_revenue
SET revenue = 0
WHERE order_date < '2026-01-01';

-- Grants must be applied to the clone explicitly.
GRANT USAGE ON DATABASE analytics_dev TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics_dev.marts TO ROLE analytics_engineer;

DROP DATABASE analytics_dev;
```

Zero-copy clones share micro-partitions until modified, which makes production-sized dev, test, and audit environments fast and cheap — just re-grant access, because privileges do not travel with the clone.

A clone copies the object's metadata and records a reference to each of the source's micro-partitions. Both objects read the same files until one of them changes them. Because partitions are immutable, a change on either side writes a new partition for the affected rows rather than editing a shared one, so the two objects diverge only where they differ. This is copy-on-write at partition granularity, not at row granularity.

A 100 TB database clones in seconds and adds almost no storage. If the clone rewrites 5 percent of its partitions during a test, it adds roughly 5 percent of the data, not 100 percent. A clone that is modified everywhere converges toward the cost of a full copy.

Cloning shares storage, so dropping the source while the clone exists frees nothing. A clone also starts its own Time Travel history when it is created; it does not inherit the source's history, so clone again if you need a later retention window.

## What are streams and how do they capture CDC?

```sql
CREATE OR REPLACE STREAM raw.orders.orders_stream ON TABLE raw.orders.orders;

-- Inspect the pending delta without consuming it.
SELECT METADATA$ACTION, METADATA$ISUPDATE, METADATA$ROW_ID,
       order_id, status, updated_at
FROM raw.orders.orders_stream
LIMIT 10;

-- Consuming the stream in DML advances the offset atomically with the write.
MERGE INTO analytics.orders AS t
USING (
  SELECT order_id, status, updated_at, METADATA$ACTION AS action
  FROM raw.orders.orders_stream
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1
) AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED AND s.action = 'INSERT'
  THEN INSERT (order_id, status, updated_at) VALUES (s.order_id, s.status, s.updated_at);

-- Check whether a stream has pending data before starting work.
SELECT SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream');
SHOW STREAMS LIKE 'ORDERS_STREAM' IN SCHEMA raw.orders;
```

A stream is a transactional change cursor over a table: it only advances when consumed by DML, it expires if you leave it unread past the retention window, and it hands you a delta that you must merge idempotently.

A stream does not copy rows; it holds an offset into the source table's change history and a state flag. When a DML statement reads the stream and writes the result, the offset advances inside the same transaction, so a rollback leaves the stream pointing at the same changes. A `SELECT` from the stream alone never advances it, no matter how many times it runs.

Standard streams capture inserts, updates, and deletes, exposing `METADATA$ACTION`, `METADATA$ISUPDATE`, and `METADATA$ROW_ID`. Append-only streams capture inserts only and do less bookkeeping, so they are cheaper for immutable event tables where rows are never updated or deleted. Choose append-only unless you genuinely need update and delete tracking.

A stream goes stale when the source's data retention window passes a point it still needs. `MAX_DATA_EXTENSION_TIME_IN_DAYS` (14 by default) extends retention for unconsumed streams, and once a stream is stale it cannot be recovered: it must be dropped and recreated, losing every change in between.

In [ ]:
class Stream:
    """A Snowflake-style stream: an offset into the source table's change log."""

    def __init__(self, con, change_log: str):
        self.con, self.change_log, self.offset = con, change_log, 0

    def pending(self) -> list[tuple]:  # SELECT * FROM stream: never advances the offset
        return self.con.execute(f"SELECT * FROM {self.change_log} WHERE seq > ? ORDER BY seq",
                                [self.offset]).fetchall()

    def consume_into(self, merge_sql: str) -> None:  # DML that reads the stream advances it atomically
        high = self.con.execute(f"SELECT coalesce(max(seq), 0) FROM {self.change_log}").fetchone()[0]
        self.con.execute("BEGIN")
        try:
            self.con.execute(merge_sql, [self.offset, high])
            self.con.execute("COMMIT")
            self.offset = high
        except Exception:
            self.con.execute("ROLLBACK")  # offset unchanged: the delta is retried, not lost
            raise


con.execute("""
    CREATE TABLE orders_changes (seq INTEGER, action VARCHAR, order_id INTEGER, status VARCHAR, updated_at INTEGER);
    INSERT INTO orders_changes VALUES
        (1, 'INSERT', 10, 'created', 100), (2, 'INSERT', 11, 'created', 101),
        (3, 'INSERT', 10, 'shipped', 105),  -- same key changed twice in one batch
        (4, 'DELETE', 11, NULL, 106);
    CREATE TABLE analytics_orders (order_id INTEGER, status VARCHAR, updated_at INTEGER);
""")
MERGE_FROM_STREAM = """
    MERGE INTO analytics_orders t
    USING (SELECT * FROM orders_changes WHERE seq > ? AND seq <= ?
           QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY seq DESC) = 1) s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
    WHEN MATCHED THEN UPDATE SET status = s.status, updated_at = s.updated_at
    WHEN NOT MATCHED AND s.action = 'INSERT' THEN INSERT VALUES (s.order_id, s.status, s.updated_at)
"""
stream = Stream(con, "orders_changes")
assert len(stream.pending()) == len(stream.pending()) == 4  # reading twice does not consume
stream.consume_into(MERGE_FROM_STREAM)
print(con.execute("SELECT * FROM analytics_orders").fetchall(), "| pending:", stream.pending())
assert con.execute("SELECT * FROM analytics_orders").fetchall() == [(10, "shipped", 105)]
assert stream.pending() == []

## What are tasks and how do you build incremental pipelines?

```sql
CREATE OR REPLACE TASK raw.orders.load_task
  WAREHOUSE = elt_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream')
AS CALL analytics.load_orders_incremental();

CREATE OR REPLACE TASK analytics.transform_task
  WAREHOUSE = transform_wh
  AFTER raw.orders.load_task
AS CALL analytics.build_orders_marts();

-- Child tasks first, then the root.
ALTER TASK analytics.transform_task RESUME;
ALTER TASK raw.orders.load_task RESUME;

-- Serverless alternative: no warehouse to keep warm.
CREATE OR REPLACE TASK analytics.compact_task
  SCHEDULE = 'USING CRON 0 3 * * * America/Chicago'
AS INSERT INTO analytics.audit_log SELECT CURRENT_TIMESTAMP(), 'compacted';

-- Observe runs, durations, and errors.
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(information_schema.task_history(
  scheduled_time_range_start => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY scheduled_time DESC
LIMIT 50;
```

Tasks chain SQL into a DAG and, gated on `SYSTEM$STREAM_HAS_DATA`, give you an incremental pipeline inside Snowflake — make every step idempotent, because a failed subtree does not roll back its parents.

A standalone task has a schedule; a child task has `AFTER` and no schedule. A run of the root cascades to children as predecessors succeed. Task graphs are versioned, so a run completes against the definition that existed when it started and editing a task mid-run does not change an executing graph. `EXECUTE TASK <name>` triggers a manual run of a standalone task or the root of a graph, independent of the schedule; it runs the root even when suspended but does not resume suspended child tasks.

Serverless tasks omit `WAREHOUSE` and run on Snowflake-managed compute, billed as serverless credits visible in `SERVERLESS_TASK_HISTORY`; warehouse-backed tasks consume the named warehouse instead and can be gated with `WHEN SYSTEM$STREAM_HAS_DATA(...)` so an idle schedule costs nothing. Scheduled tasks accept a plain interval or a cron expression with a time zone.

Because a graph is not one transaction, a failed child leaves its committed predecessors applied and a retry re-runs only from the failed point. Every task body should therefore be idempotent, and overlap should be handled deliberately so a slow run and the next scheduled run do not fight over the same target rows.

## What are dynamic tables?

```sql
CREATE OR REPLACE DYNAMIC TABLE analytics.daily_revenue
  TARGET_LAG = '10 minutes'
  WAREHOUSE = transform_wh
AS
  SELECT o.order_date, c.segment,
         SUM(o.amount) AS revenue,
         COUNT(*) AS orders
  FROM analytics.orders o
  JOIN analytics.customers c USING (customer_id)
  GROUP BY o.order_date, c.segment;

-- Chained dynamic table: DOWNSTREAM lets Snowflake schedule the whole chain.
CREATE OR REPLACE DYNAMIC TABLE analytics.segment_revenue
  TARGET_LAG = '10 minutes'
  WAREHOUSE = transform_wh
AS
  SELECT segment, SUM(revenue) AS revenue
  FROM analytics.daily_revenue
  GROUP BY segment;

SELECT name, target_lag_sec, scheduling_state, latest_data_timestamp
FROM TABLE(information_schema.dynamic_tables());

-- Actual refresh duration and state per run.
SELECT name, state, refresh_start_time, refresh_end_time, data_timestamp
FROM TABLE(information_schema.dynamic_table_refresh_history(
  name => 'analytics.daily_revenue',
  data_timestamp_start => DATEADD('day', -1, CURRENT_TIMESTAMP())));
```

Dynamic tables let you declare the result and a `TARGET_LAG` and let Snowflake schedule the refresh, which is powerful for SQL-shaped pipelines but hides cost — always watch refresh history and the lag you actually achieve.

A dynamic table stores its own data and keeps a refresh history. The default `REFRESH_MODE = AUTO` lets Snowflake choose incremental refresh when the query supports it and a full refresh when it does not; `INCREMENTAL` or `FULL` can be forced, and a full refresh recomputes the whole result. `TARGET_LAG = DOWNSTREAM` means the table has no independent freshness promise and is scheduled to keep its consumers on time.

Dynamic tables form a dependency graph, so adding a new downstream table does not require editing upstream schedules. That managed graph is the main operational advantage over streams plus tasks, where every new consumer means another task to create, wire, and resume in the right order.

`TARGET_LAG` is a target, not a guarantee. A refresh that takes longer than the lag, or a query that cannot be incrementalized and must fully recompute, leaves the table reliably behind. Watch `DYNAMIC_TABLE_REFRESH_HISTORY` for refresh duration and state, and compare the achieved data timestamp against the requested lag.

## What is Snowpipe and when do you use it?

```sql
-- File-based continuous ingestion with cloud event notifications.
CREATE OR REPLACE PIPE raw.orders.orders_pipe AUTO_INGEST = TRUE
AS COPY INTO raw.orders.orders
   FROM @raw.orders.orders_landing
   FILE_FORMAT = (TYPE = PARQUET)
   MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

-- Pipe state and the notification channel used by AUTO_INGEST.
SELECT SYSTEM$PIPE_STATUS('raw.orders.orders_pipe');
SHOW PIPES LIKE 'ORDERS_PIPE' IN SCHEMA raw.orders;

-- What Snowpipe loaded recently, and what failed.
SELECT file_name, row_count, error_count, status, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('hour', -24, CURRENT_TIMESTAMP())));

-- Serverless credits consumed by pipes.
SELECT pipe_name, SUM(credits_used) AS credits
FROM snowflake.account_usage.pipe_usage_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY pipe_name
ORDER BY credits DESC;
```

Use Snowpipe for continuous file arrival, scheduled `COPY INTO` for large predictable batches, and Snowpipe Streaming for row-level latency — but plan for at-least-once delivery and avoid thousands of tiny files.

A pipe is a stored `COPY INTO` plus a load-history state. With `AUTO_INGEST = TRUE`, Snowflake reports a notification channel (an SQS queue, a Pub/Sub subscription, or an Event Grid endpoint) that you register with the bucket; an object-created event then calls the pipe, which loads just that file. Snowflake records each file in load metadata so a redelivered event does not load it again.

Snowpipe runs on Snowflake-managed compute billed as serverless credits, and every file carries its own event, metadata, and processing overhead, so thousands of tiny files multiply both latency and cost. Snowpipe Streaming avoids files entirely: a client writes rows through the API and Snowflake commits them into the table, trading per-file overhead for row-level latency.

Delivery is at-least-once. A file or a row can be processed twice, so the target table or the downstream merge must key on a business key or the file name. The pipe will not make duplicates safe for you.

## What are stages and how do you load with COPY INTO?

```sql
-- External stage over your own bucket, credential-free via a storage integration.
CREATE OR REPLACE STORAGE INTEGRATION s3_orders_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('s3://acme-raw/orders/');

CREATE OR REPLACE STAGE raw.orders.landing
  URL = 's3://acme-raw/orders/'
  STORAGE_INTEGRATION = s3_orders_int
  FILE_FORMAT = (TYPE = PARQUET);

-- Load one day of files, tolerate bad rows, and clean up after success.
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Dry run: report errors without loading anything.
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET)
  VALIDATION_MODE = 'RETURN_ERRORS';

-- Unload a result set, splitting it into many files.
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue WHERE order_date = '2026-10-04')
FILE_FORMAT = (TYPE = PARQUET)
MAX_FILE_SIZE = 100000000
HEADER = TRUE;

-- Load history for the last day, including errors.
SELECT file_name, status, row_count, error_count, first_error_message, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY last_load_time DESC;
```

Stages abstract storage locations and `COPY INTO` loads or unloads in parallel — feed it 100 to 250 MB compressed files, use `PATTERN`, `ON_ERROR`, and `VALIDATION_MODE` deliberately, and watch `COPY_HISTORY` for failures.

Load metadata is Snowflake's memory of which files it has already loaded into a table, and it lasts 64 days. `FORCE = FALSE` consults it and skips a file already loaded; `FORCE = TRUE` ignores it. The memory belongs to the table, so the same file loaded into a second table is loaded again.

`ON_ERROR` gives you a policy per statement: `ABORT_STATEMENT` fails the load, `CONTINUE` loads good rows and records bad ones, and `SKIP_FILE` skips a file containing an error. For Snowpipe the policy lives in the pipe definition, and every outcome lands in `COPY_HISTORY` with an `error_count` and a status, which is the first place to look when data never appears.

File size drives parallelism: compressed files of roughly 100 to 250 MB let the warehouse keep many files in flight at once, while one multi-gigabyte file limits how much of the warehouse can be used. Unloading has the same property, which is what `MAX_FILE_SIZE` is for.

## How do you handle semi-structured data with VARIANT?

```sql
CREATE OR REPLACE TABLE raw.events (event_id STRING, payload VARIANT);

COPY INTO raw.events (event_id, payload)
FROM (
  SELECT $1:event_id::STRING, $1
  FROM @raw.events.event_stage
)
FILE_FORMAT = (TYPE = JSON STRIP_OUTER_ARRAY = TRUE);

-- Explore before you flatten.
SELECT payload:event_type::STRING AS event_type, TYPEOF(payload:amount) AS amount_type,
       OBJECT_KEYS(payload) AS keys
FROM raw.events
LIMIT 5;

-- Query typed fields with a safe cast.
SELECT payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount
FROM raw.events
WHERE payload:event_type::STRING = 'purchase';

-- Explode an array into rows.
SELECT e.event_id, i.value:sku::STRING AS sku, i.value:qty::NUMBER AS qty
FROM raw.events e,
     LATERAL FLATTEN(input => e.payload:items) i;

-- Promote the fields people actually use into typed columns.
CREATE OR REPLACE VIEW analytics.v_events AS
SELECT event_id,
       payload:event_type::STRING AS event_type,
       payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount,
       payload:ts::TIMESTAMP_NTZ AS event_ts
FROM raw.events;
```

Keep raw `VARIANT` for replay and drift, use `:` with `TRY_CAST` and `FLATTEN` to explore it, and promote the fields you actually query into typed columns so pruning and joins behave predictably.

When a document is loaded, Snowflake parses it and writes each distinct path into an internal columnar representation, so a key repeated across millions of rows is stored once as a path with its values beside it rather than as text in every row. Statistics exist for variant paths, but they are weaker than on typed columns, which is why a predicate on `payload:user.id` prunes worse than one on a typed `user_id`.

Path syntax uses `:` for object keys and `[]` for array elements, and both accept a constant or a computed key. Keys containing spaces or special characters need double quotes or bracket syntax, for example `payload:"user id"` or `payload['user id']`. `FLATTEN` turns an array into rows and exposes `VALUE`, `PATH`, `INDEX`, and `THIS`.

`TRY_CAST` returns null both when a value is missing and when it is malformed, so a quiet null can hide a schema change. When a path is business-critical, validate it with `TYPEOF` or check `OBJECT_KEYS`, and promote it into a typed column instead of recasting it in every query.

In [ ]:
con.execute("""
    CREATE TABLE raw_events AS SELECT event_id, payload::JSON AS payload FROM (VALUES
        ('e1', '{"event_type": "purchase", "user": {"id": 42}, "amount": "19.90",
                 "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}'),
        ('e2', '{"event_type": "purchase", "user": {"id": 7}, "amount": "n/a", "items": []}'),
        ('e3', '{"event_type": "view", "user": {"id": 42}}')
    ) t(event_id, payload)
""")
# payload:user.id::NUMBER and TRY_CAST(payload:amount AS NUMBER(12,2))
typed = con.execute("""
    SELECT event_id, CAST(payload ->> '$.user.id' AS INTEGER) AS user_id,
           TRY_CAST(payload ->> '$.amount' AS DECIMAL(12, 2)) AS amount
    FROM raw_events WHERE payload ->> '$.event_type' = 'purchase' ORDER BY event_id
""").fetchall()
# LATERAL FLATTEN(input => payload:items)
items = con.execute("""
    SELECT e.event_id, i ->> '$.sku' AS sku, CAST(i ->> '$.qty' AS INTEGER) AS qty
    FROM raw_events e, UNNEST(CAST(payload -> '$.items' AS JSON[])) AS u(i)
    ORDER BY sku
""").fetchall()
print(typed)
print(items)
assert typed[1][2] is None  # TRY_CAST: a malformed value becomes a quiet NULL
assert items == [("e1", "A", 2), ("e1", "B", 1)]

## How do external tables work?

```sql
CREATE OR REPLACE EXTERNAL TABLE raw.lake.events_ext (
  event_id STRING AS (VALUE:c1::STRING),
  payload  VARIANT AS (VALUE)
)
LOCATION = @raw.lake.events_stage/events/
FILE_FORMAT = (TYPE = PARQUET)
AUTO_REFRESH = TRUE;

-- Manual refresh when auto-refresh is not configured.
ALTER EXTERNAL TABLE raw.lake.events_ext REFRESH;

-- Query in place, with file metadata for lineage.
SELECT METADATA$FILENAME, METADATA$FILE_ROW_NUMBER, event_id, payload:event_type::STRING
FROM raw.lake.events_ext
WHERE event_id IS NOT NULL
LIMIT 10;

-- Seen files and their partitions, useful when pruning looks wrong.
SELECT COUNT(*) AS files, MIN(METADATA$FILENAME) AS sample_file
FROM raw.lake.events_ext;
```

External tables query your lake in place with no load and no copy, but they are read-only and depend on file layout, so keep hot, joined data native and leave cold history external.

An external table holds no row data in Snowflake: only column definitions and a catalog of the files and partitions found in the stage. At query time the warehouse opens the underlying objects directly, using path-derived partition columns and the statistics in each file's footer to choose files. Pruning therefore depends on directory layout and on the source format's own statistics, not on Snowflake micro-partitions.

Because the files live outside Snowflake, features tied to storage internals do not apply: external tables are read-only, have no Time Travel, cannot be clustered, and cannot be written by DML. `AUTO_REFRESH = TRUE` picks up new files only when the stage has event notifications; otherwise the file list stays stale until `ALTER EXTERNAL TABLE ... REFRESH` runs.

A layout that puts every date in one directory removes the only cheap pruning signal. Naming prefixes by partition, such as `event_date=2026-10-04/`, is what lets a date filter avoid opening unrelated files.

## How do Snowflake Iceberg tables work?

```sql
CREATE OR REPLACE EXTERNAL VOLUME iceberg_vol
  STORAGE_LOCATIONS = ((
    NAME = 's3_primary'
    STORAGE_PROVIDER = 'S3'
    STORAGE_BASE_URL = 's3://acme-iceberg/'
    STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::111122223333:role/snowflake-iceberg'
  ));

-- Snowflake-managed: Snowflake writes and maintains the Iceberg metadata.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_iceberg (
  order_id NUMBER,
  order_date DATE,
  amount NUMBER(12, 2)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'iceberg_vol'
BASE_LOCATION = 'analytics/orders';

-- Catalog-linked: an external catalog owns the table.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_glue
  CATALOG = 'glue_catalog_integration'
  CATALOG_TABLE_NAME = 'orders'
  EXTERNAL_VOLUME = 'iceberg_vol';

INSERT INTO analytics.orders_iceberg
SELECT order_id, order_date, amount FROM raw.orders_staging;

SELECT COUNT(*) AS rows, COUNT(DISTINCT order_date) AS days
FROM analytics.orders_iceberg;
```

Iceberg tables put Snowflake on the same open Parquet-and-metadata table as Spark and Trino, so choose deliberately between a Snowflake-managed table and a catalog-linked one, and make write ownership explicit.

An Iceberg table is a set of Parquet data files plus a metadata tree: manifests list data files, a manifest list lists manifests, and a snapshot points at one manifest list. A writer commits by adding a new snapshot atomically, so two engines can share the table without coordinating on files, as long as only one writer owns a given table at a time.

Snowflake needs an external volume to reach the storage layer even for a catalog-linked table. A Snowflake-managed table (`CATALOG = 'SNOWFLAKE'`) is written through Snowflake, which maintains the metadata and can apply Snowflake features. A catalog-linked table is registered from an external catalog that owns the metadata, and Snowflake is generally a reader there.

Openness moves responsibility to you. File compaction, snapshot expiration, and storage lifecycle are no longer handled by a warehouse-native format, and cross-engine writers can commit conflicting snapshots, so designate a single writer per table.

## What is Snowpark and how do you use Python in Snowflake?

```python
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_

session = Session.builder.configs(connection_parameters).create()  # credentials from a secrets store

orders = session.table("RAW.ORDERS.ORDERS")
daily = orders.group_by(col("ORDER_DATE")).agg(sum_(col("AMOUNT")).alias("REVENUE"))
daily.write.mode("overwrite").save_as_table("ANALYTICS.MARTS.DAILY_REVENUE")  # pushed down as SQL
```

Snowpark lets you express transformations as a lazy DataFrame plan that Snowflake executes next to the data, so keep work server-side, prefer native functions over row-by-row UDFs, and use a Snowpark-optimized warehouse only for memory-bound jobs.

Snowpark does not run Python on the warehouse for a DataFrame expression. The client builds a logical plan, which cloud services compiles to SQL and hands to the warehouse; only code inside a user-defined function or a stored procedure actually executes in a Python runtime. That is why DataFrame transforms perform like the SQL they compile to, while a Python UDF over the same rows does not.

UDFs come in two shapes. A scalar Python UDF is invoked once per row and pays interpreter overhead on every call. A vectorized pandas UDF receives batches as pandas Series and returns a Series, amortizing that overhead across many rows; reach for it when the logic cannot be expressed in native SQL.

A Snowpark-optimized warehouse has more memory and local SSD per node for memory-heavy UDFs and model scoring, and it costs more per credit than a standard warehouse. Use it when a job genuinely spills or needs the memory, not as a default.

## What are UDFs and stored procedures?

```sql
-- A SQL scalar UDF: pure, composable, and safe inside a large scan.
CREATE OR REPLACE FUNCTION analytics.fiscal_quarter(d DATE)
  RETURNS STRING
  LANGUAGE SQL
AS
$$
  CASE WHEN MONTH(d) >= 2 THEN 'Q' || CEIL((MONTH(d) - 1) / 3.0)::INT
       ELSE 'Q4' END
$$;

SELECT analytics.fiscal_quarter(order_date) AS fq, SUM(amount)
FROM analytics.orders
GROUP BY fq;

-- A Snowflake Scripting procedure: multiple statements and control flow.
CREATE OR REPLACE PROCEDURE analytics.load_daily(load_date DATE)
  RETURNS STRING
  LANGUAGE SQL
AS
$$
BEGIN
  DELETE FROM analytics.marts.daily_revenue WHERE order_date = :load_date;
  INSERT INTO analytics.marts.daily_revenue
  SELECT order_date, SUM(amount)
  FROM raw.orders.orders
  WHERE order_date = :load_date
  GROUP BY order_date;
  RETURN 'loaded ' || :load_date::STRING;
END;
$$;

CALL analytics.load_daily('2026-10-04');
```

UDFs extend SQL with a value per row and cannot query tables, while stored procedures orchestrate multi-statement work — keep UDFs pure and small, and remember that a row-by-row Python UDF is far slower than a native SQL expression.

A scalar UDF must be side-effect free and cannot query tables, which lets the optimizer inline or parallelize it across a scan. A stored procedure is different: it is invoked with `CALL`, runs its statements in order with the privileges of its owner role by default, and can contain DML, DDL, and control flow. A procedure's statements are not one automatic transaction; only what you wrap in an explicit transaction commits or rolls back together.

Python UDFs and procedures are packaged with a runtime version, a handler naming the callable, optional `PACKAGES`, and optional `IMPORTS` for staged files. Snowflake stages the code and loads it into the execution environment, so the artifact must be uploaded before `CREATE FUNCTION` and re-uploaded when the code changes.

A procedure is the right place to coordinate steps, not to process rows. Looping over rows in a procedure multiplies statement overhead and defeats set-based execution; keep the loop inside one statement, or batch the work so each call handles many rows.

## What is the RBAC role and privilege model?

```sql
USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS analytics_engineer;

USE ROLE SYSADMIN;
GRANT USAGE ON WAREHOUSE transform_wh TO ROLE analytics_engineer;
GRANT USAGE ON DATABASE analytics TO ROLE analytics_engineer;
GRANT USAGE ON SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON FUTURE TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT ROLE analytics_engineer TO ROLE SYSADMIN;

-- Give a person the role, not the privileges.
GRANT ROLE analytics_engineer TO USER analyst_jane;

-- Audit who has what.
SHOW GRANTS TO ROLE analytics_engineer;
SELECT * FROM snowflake.account_usage.grants_to_roles
WHERE role LIKE 'ANALYTICS%' AND deleted_on IS NULL;
```

Grant privileges to functional roles, grant roles to roles and users, cover new objects with future grants, and keep `ACCOUNTADMIN` and `OWNERSHIP` deliberate rather than routine.

The predefined roles divide responsibility. `ACCOUNTADMIN` holds account-level settings and can see everything; `SYSADMIN` owns warehouses and databases; `SECURITYADMIN` manages grants and can monitor any object; `USERADMIN` creates users and roles; `PUBLIC` is granted to every user automatically. `SYSADMIN` and `SECURITYADMIN` are granted to `ACCOUNTADMIN`, which is why `ACCOUNTADMIN` inherits broad access rather than being granted it object by object.

A future grant is stored on the schema and applied to objects created later; it is not retroactive and does not reach objects that already exist. A deployment that creates a new table still needs the future grant to have been issued earlier, or an explicit grant afterward. Ownership is separate from privileges: exactly one role owns an object, and `OWNERSHIP` implies every privilege on it.

Because `ACCOUNTADMIN` inherits `SYSADMIN`, running applications or scheduled tasks as `ACCOUNTADMIN` makes least-privilege auditing impossible. Transfer ownership to custom roles, grant those roles to `SYSADMIN`, and keep `ACCOUNTADMIN` for account administration only.

## What are secure views, row access policies, and column masking?

```sql
CREATE OR REPLACE SECURE VIEW analytics.v_orders AS
SELECT order_id, customer_id, amount
FROM analytics.orders
WHERE order_status <> 'test';

CREATE OR REPLACE ROW ACCESS POLICY security.region_policy
  AS (region STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() = 'ACCOUNTADMIN'
    OR EXISTS (
      SELECT 1 FROM security.region_map m
      WHERE m.role_name = CURRENT_ROLE() AND m.region = region
    );
ALTER TABLE analytics.orders
  ADD ROW ACCESS POLICY security.region_policy ON (region);

CREATE OR REPLACE MASKING POLICY security.email_mask
  AS (val STRING) RETURNS STRING ->
    CASE WHEN CURRENT_ROLE() IN ('PII_ADMIN', 'ACCOUNTADMIN') THEN val
         ELSE REGEXP_REPLACE(val, '^[^@]+', '***') END;
ALTER TABLE analytics.customers
  MODIFY COLUMN email SET MASKING POLICY security.email_mask;

-- See which policies protect which objects.
SHOW ROW ACCESS POLICIES;
SHOW MASKING POLICIES;
```

Secure views, row access policies, and masking policies enforce least privilege in the engine rather than in each query, keyed on `CURRENT_ROLE()` — and row and column policies require Enterprise Edition.

Row access and masking policies are evaluated at query time against the role active in the session, not stored per user or per tool. The same SQL run by two roles can return different rows or different column values, which is why enforcement survives a client that forgets to add a filter. Because a policy is a function, it must be deterministic enough for the engine to plan and is called as part of the scan.

Order matters conceptually. The row access policy decides which rows exist for the role, and the masking policy decides what the returned values look like for the rows that survive. A masking policy can use `CURRENT_ROLE()` and `CURRENT_USER()` to choose who sees the real value, while a row access policy usually compares the role against a mapping table. `CURRENT_ROLE()` is the safer key, because roles are the unit of governance and users change.

Both features require Enterprise Edition. A policy that calls a mapping table adds work to every query that touches the protected object, so keep the mapping small and the logic simple. A policy that returns null instead of false can behave differently from a false, so be explicit with boolean expressions.

## How do resource monitors control cost?

```sql
CREATE OR REPLACE RESOURCE MONITOR elt_monitor WITH CREDIT_QUOTA = 2000
  FREQUENCY = MONTHLY
  TRIGGERS
    ON 75 PERCENT DO NOTIFY
    ON 100 PERCENT DO SUSPEND
    ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE elt_wh SET RESOURCE_MONITOR = elt_monitor;

-- Account-level guardrail for warehouses without their own monitor.
CREATE OR REPLACE RESOURCE MONITOR account_monitor WITH CREDIT_QUOTA = 20000
  FREQUENCY = MONTHLY
  TRIGGERS ON 90 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER ACCOUNT SET RESOURCE_MONITOR = account_monitor;

-- Credits actually consumed, per warehouse and day.
SELECT warehouse_name, DATE_TRUNC('day', start_time) AS usage_day,
       SUM(credits_used) AS credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -14, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY usage_day DESC, credits DESC;

SHOW RESOURCE MONITORS;
```

A resource monitor turns cost from an observation into an enforced limit — notify early, suspend before an emergency, and set quotas and per-warehouse monitors from measured usage rather than guesswork.

A monitor totals the credits used by its assigned warehouses plus the cloud services that supported them, within an interval that resets at 12:00 AM UTC. Only `ACCOUNTADMIN` can create one, a warehouse can belong to at most one warehouse monitor, and an account monitor does not override the assignment of individual warehouses: either monitor can trigger a suspend.

The important exemptions are serverless features and cloud services. An account-level resource monitor does not control Snowflake-managed compute for Snowpipe, automatic clustering, or materialized views, and budgets exist for those. A warehouse-level monitor can observe cloud-services credits but cannot suspend them, and a suspension clears when the interval resets, the quota is raised, or the warehouse is unassigned.

Monitor thresholds are calculated on all consumed cloud-services credits, ignoring the 10 percent daily adjustment, so a monitor can report 100 percent of quota while the actual bill is smaller. If you need a hard guarantee, set the suspend threshold below quota, for example 90 percent, because suspension takes effect slightly after the threshold is crossed.

## How do transactions and isolation work in Snowflake?

```sql
ALTER SESSION SET TRANSACTION_ISOLATION_LEVEL = 'SERIALIZABLE';

BEGIN;
UPDATE analytics.accounts SET balance = balance - 100 WHERE account_id = 1;
UPDATE analytics.accounts SET balance = balance + 100 WHERE account_id = 2;
COMMIT;

-- DDL auto-commits independently: rolling back does not undo the CREATE.
BEGIN;
CREATE TABLE analytics.tmp_check (id NUMBER);
ROLLBACK;
SHOW TABLES LIKE 'TMP_CHECK' IN SCHEMA analytics;
```

Snowflake gives ACID transactions with read-committed snapshot isolation and non-blocking readers, and SERIALIZABLE when you need it — just remember that DDL auto-commits outside your transaction.

The default isolation is read committed at the statement level: each statement takes a fresh snapshot when it begins, so two statements in the same explicit transaction can see different versions if a commit lands between them. Readers do not block writers and writers do not block readers, because a reader simply addresses the versions in its snapshot. Concurrent DML against the same table is serialized through the transaction manager rather than through a visible lock wait.

Outside an explicit transaction every statement is its own transaction and commits on success; that is autocommit. `BEGIN` opens a transaction that stays open until `COMMIT` or `ROLLBACK`, and DDL inside it still commits immediately and independently. `SERIALIZABLE` changes conflict handling: instead of letting statements interleave, Snowflake aborts one of the conflicting transactions, so the application must be prepared to retry.

A long open transaction keeps the versions it can still read alive, which delays storage cleanup and can consume the Time Travel window. Keep transactions short and put long computation outside them; wrap only the statements that must be atomic.

## What is Snowflake data sharing?

```sql
-- Provider side: grant privileges to the share, then add consumers.
CREATE SHARE sales_share;
GRANT USAGE ON DATABASE analytics TO SHARE sales_share;
GRANT USAGE ON SCHEMA analytics.marts TO SHARE sales_share;
GRANT SELECT ON TABLE analytics.marts.fct_orders TO SHARE sales_share;
ALTER SHARE sales_share ADD ACCOUNTS = consumer_org.consumer_acct;

-- Consumer side: mount the share and use it.
CREATE DATABASE shared_sales FROM SHARE provider_org.provider_acct.sales_share;
GRANT IMPORTED PRIVILEGES ON DATABASE shared_sales TO ROLE analyst;

SELECT COUNT(*) FROM shared_sales.marts.fct_orders;

-- Cross-region sharing: replicate first, then share from the secondary.
ALTER DATABASE analytics ENABLE REPLICATION TO ACCOUNTS secondary_org.secondary_acct;
CREATE DATABASE analytics_replica FROM SHARE secondary_org.secondary_acct.sales_share;
```

Data sharing gives consumers live, read-only access to your tables with no copy and no ETL, so govern the grants like a product and replicate when you need to cross a region or cloud.

A share is a named set of grants on objects in a database. When a consumer creates a database from the share, they receive a read-only mount that resolves to the provider's storage and metadata; no rows are copied at share time and no synchronization job exists. New rows committed by the provider are visible to the consumer on their next query, bounded by the provider's grants and any Time Travel the provider applies.

Consumers need `IMPORTED PRIVILEGES` on the mounted database and pay for the compute that reads it. Cross-region or cross-cloud sharing requires replicating the database first, because a share is region-scoped, and replication is serverless and billed separately. Reader accounts exist for consumers without their own Snowflake account and are created and managed by the provider.

A share can expose object definitions as well as data, so treat it as a product with a schema contract. Replacing or dropping the shared database breaks every consumer; coordinate changes and prefer additive schema evolution.

## What is the search optimization service?

```sql
-- Enable on selected columns and methods, not the whole table.
ALTER TABLE analytics.customers
  ADD SEARCH OPTIMIZATION ON EQUALITY(email, customer_id);
ALTER TABLE analytics.events
  ADD SEARCH OPTIMIZATION ON SUBSTRING(url);

-- These now use the optimized structure instead of a full scan.
SELECT customer_id, email FROM analytics.customers WHERE email = 'someone@example.com';
SELECT event_id FROM analytics.events WHERE url LIKE '%/checkout/complete%';

-- Inspect the configured optimization, then remove it if it stops paying.
SHOW SEARCH OPTIMIZATION ON analytics.customers;
ALTER TABLE analytics.customers DROP SEARCH OPTIMIZATION ON EQUALITY(email);
```

Search optimization makes selective point lookups and leading-wildcard searches fast where pruning cannot help, but it is maintained on every DML — enable it on large, slowly changing tables and measure against its serverless cost.

Search optimization maintains a persistent access path alongside the table for specific columns and predicate methods. The optimizer rewrites a matching predicate to use that structure instead of scanning micro-partitions, so the query keeps its original text. Maintenance is asynchronous: after DML the structure catches up in the background, so improvements are not instantaneous.

It targets predicates that min/max pruning cannot narrow: equality and `IN` on a column, a `LIKE` with a leading wildcard, `ARRAY_CONTAINS`, and geospatial predicates. It does not speed up aggregations, joins, or broad range scans, and enabling it changes no results; it only gives the optimizer another access path.

Cost is proportional to change volume, because every DML adds maintenance work, and on a high-churn staging table the maintenance credits can exceed anything saved. Enable it on large, slowly changing lookup tables, verify in the query profile that the access path is used, and remove it when those queries disappear.

## What are materialized views?

```sql
CREATE OR REPLACE MATERIALIZED VIEW analytics.mv_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue, COUNT(*) AS orders
FROM analytics.orders
GROUP BY order_date;

-- The optimizer can rewrite this to read the materialized view.
SELECT order_date, SUM(amount) FROM analytics.orders GROUP BY order_date;

SHOW MATERIALIZED VIEWS IN SCHEMA analytics;

-- Pause maintenance (it may become stale), then resume.
ALTER MATERIALIZED VIEW analytics.mv_daily_revenue SUSPEND;
ALTER MATERIALIZED VIEW analytics.mv_daily_revenue RESUME;

-- Serverless credits spent maintaining it.
SELECT name, SUM(credits_used) AS credits
FROM snowflake.account_usage.materialized_view_refresh_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY name
ORDER BY credits DESC;
```

Materialized views are managed, incrementally maintained single-table aggregates that the optimizer can substitute into queries, so reserve them for repeated expensive aggregations and watch maintenance credits.

A materialized view stores its result as a Snowflake-managed internal table and is maintained from the base table's change tracking. Because maintenance must be incremental, Snowflake restricts the definition to a single base table with no joins, no `HAVING`, no window functions, and no user-defined functions, and only a subset of aggregates and scalar expressions is allowed.

The optimizer can transparently rewrite a query against the base table to read the materialized view when it can prove the result is equivalent, which is why application SQL does not change. Maintenance runs on serverless compute, and `ALTER MATERIALIZED VIEW ... SUSPEND` stops updates, after which the view may become stale until it is resumed.

A rewrite is not guaranteed. Predicates or expressions the optimizer cannot map to the view's grouping, and joins against the base table, may prevent substitution even when a human can see the equivalence. Confirm the rewrite in the query profile and watch maintenance credits, because a high-churn base table can cost more to maintain than the view saves.

## What is the query acceleration service?

```sql
ALTER WAREHOUSE bi_wh SET
  ENABLE_QUERY_ACCELERATION = TRUE
  QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;

-- Would this specific query benefit?
SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>');

-- Which queries actually used acceleration, and how much did they scan?
SELECT query_id, warehouse_name,
       query_acceleration_bytes_scanned,
       query_acceleration_time,
       total_elapsed_time / 1000 AS elapsed_s
FROM snowflake.account_usage.query_history
WHERE query_acceleration_bytes_scanned > 0
  AND start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
ORDER BY query_acceleration_bytes_scanned DESC
LIMIT 20;

SHOW WAREHOUSES LIKE 'BI_WH';
```

Query acceleration absorbs occasional large scans with serverless compute so you do not have to size a warehouse for its worst query, but it is metered and never a substitute for good pruning and query shape.

For an eligible query the service provisions serverless compute that processes part of the scan and part of the aggregation while the warehouse handles the rest. Eligibility is decided at compile time from the query shape: scans with filters and aggregations over large tables qualify, while plans dominated by joins, sorts, or very selective lookups generally do not.

`QUERY_ACCELERATION_HISTORY` records the credits the service consumed, and `QUERY_HISTORY` exposes `QUERY_ACCELERATION_BYTES_SCANNED` and `QUERY_ACCELERATION_TIME` per query. `QUERY_ACCELERATION_MAX_SCALE_FACTOR` caps how much serverless compute can be added, which is the main guardrail on spend.

Acceleration is billed separately from the warehouse, so it can make a query faster and more expensive at the same time. It also cannot compensate for a full scan that pruning could have avoided; fix the predicate first.

## How do you migrate from Redshift, Hadoop, or another warehouse?

```sql
-- Reconcile beyond row counts: sums, distinct keys, and null rates.
SELECT 'orders' AS entity,
       COUNT(*) AS rows_loaded,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(amount) AS total_amount,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customers
FROM analytics.fct_orders;

-- Track migration progress per subject area.
SELECT subject_area, status, source_rows, target_rows,
       CASE WHEN source_rows = target_rows THEN 'ok' ELSE 'investigate' END AS check_result
FROM migration.reconciliation_log
ORDER BY subject_area;
```

Migrate data before logic, let tooling translate syntax while you own the data model, permissions, and workload placement, and validate with sums, keys, and reports — not row counts alone.

Translate in two passes: first the mechanical syntax, which tooling such as SnowConvert handles, then the semantics, which no tool decides for you. Empty string versus null, integer division and rounding, timestamp precision and time zone handling, identifier case folding, and sequence or `IDENTITY` behavior all differ across engines, and each difference needs a test with real data rather than a code review.

Physical design does not translate directly. Redshift distribution and sort keys become load ordering plus an optional clustering key; Hive partition columns become directory layout and, for external or Iceberg tables, `PARTITION BY`; secondary indexes and `ANALYZE` have no equivalent because micro-partition statistics are automatic. Warehouse sizing is a separate decision from the original cluster shape.

Unquoted identifiers are folded to uppercase in Snowflake, so code that assumed lowercase names can break even when the SQL is otherwise valid. Reconcile with sums, distinct keys, and report-level comparisons rather than row counts alone, and keep the source running until a full reporting cycle passes.

## What has changed in Snowflake recently?

Snowflake ships continuously, so a page this long goes out of date one section at a time. The changes below alter a design decision. Dynamic tables and Iceberg tables have their own sections above.

**Gen2 standard warehouses.** A newer generation of the standard warehouse, built on faster hardware with software optimizations for analytics and data engineering. If you create a standard warehouse without naming a generation, Snowflake now creates Gen2 wherever the region supports it. An existing warehouse converts with `ALTER WAREHOUSE ... SET GENERATION = '2'`. Gen2 is not offered in the X5LARGE and X6LARGE sizes. Before you convert a fleet, check the current credit rates and benchmark a representative workload.

**Cortex AI functions.** SQL functions that call hosted large language models from inside a query: `AI_COMPLETE`, `AI_CLASSIFY`, `AI_FILTER`, `AI_AGG`, `AI_EXTRACT`, `AI_SENTIMENT`, `AI_SUMMARIZE_AGG`, `AI_EMBED`, `AI_TRANSLATE`, `AI_PARSE_DOCUMENT` and others. Cost grows with the amount of text processed. Run them over a filtered, deduplicated set and persist the results instead of calling them from a dashboard query.

**Iceberg catalogs: Horizon Catalog, Open Catalog and Apache Polaris.** Apache Polaris is the open-source Iceberg REST catalog that Snowflake open-sourced, and it became an Apache top-level project in February 2026. Snowflake Open Catalog is the managed Polaris service, but Snowflake now steers new customers to Horizon Catalog. Horizon exposes an Iceberg REST API, so external engines such as Spark, Flink and Dremio can read and write Snowflake-managed Iceberg tables without a separate catalog sync.

**Hybrid tables (Unistore).** A row-oriented table type built for low-latency point reads and writes next to analytical tables. The primary key is required and enforced, unique and foreign keys are enforced, and secondary indexes are maintained synchronously. Hybrid tables are generally available in AWS and Azure commercial regions and in preview on Google Cloud. They fit small operational state such as pipeline checkpoints or application metadata, not bulk analytics, and they take more storage than standard tables.

```sql
-- Gen2 standard warehouse (the default for new standard warehouses where available)
CREATE OR REPLACE WAREHOUSE transform_wh
  WAREHOUSE_SIZE = 'MEDIUM'
  GENERATION = '2'
  AUTO_SUSPEND = 60;

-- Cortex AI function over a filtered set, results persisted once
CREATE OR REPLACE TABLE analytics.ticket_sentiment AS
SELECT ticket_id, AI_SENTIMENT(body) AS sentiment
FROM raw.support_tickets
WHERE created_at >= DATEADD(day, -1, CURRENT_TIMESTAMP());

-- Hybrid table: enforced primary key plus a secondary index
CREATE OR REPLACE HYBRID TABLE ops.pipeline_state (
  pipeline_id VARCHAR PRIMARY KEY,
  last_run_ts TIMESTAMP_NTZ,
  status      VARCHAR,
  INDEX idx_status (status)
);
```

Common mistake: reaching for a hybrid table to speed up analytics. Hybrid tables trade scan efficiency for point-lookup latency, so aggregations over millions of rows still belong in standard tables. A second is calling an AI function row by row inside a frequently refreshed view, which pays for the same text on every refresh.

## How do you tune a slow Snowflake query, step by step?

```sql
-- One query over QUERY_HISTORY with every diagnostic column that matters.
SELECT query_id,
       query_type,
       warehouse_name,
       warehouse_size,
       user_name,
       total_elapsed_time / 1000            AS elapsed_s,
       queued_overload_time / 1000          AS queue_overload_s,
       queued_provisioning_time / 1000      AS queue_provisioning_s,
       compilation_time / 1000              AS compile_s,
       execution_time / 1000                AS execution_s,
       partitions_scanned,
       partitions_total,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0), 3)      AS pruning_ratio,
       percentage_scanned_from_cache,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       ROUND(bytes_spilled_to_local_storage  / POWER(1024, 3), 2) AS gb_spill_local,
       ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_spill_remote,
       credits_used_cloud_services
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND query_type IN ('SELECT', 'INSERT', 'MERGE')
  AND execution_time > 0
ORDER BY execution_time DESC
LIMIT 50;

-- Drill into one query once you have picked the query_id.
SELECT query_id, query_text,
       total_elapsed_time / 1000 AS elapsed_s,
       execution_time / 1000 AS execution_s,
       partitions_scanned, partitions_total,
       bytes_scanned, bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';

-- A quick triage view: classify the dominant symptom per query.
SELECT query_id, warehouse_name,
       total_elapsed_time / 1000 AS elapsed_s,
       CASE
         WHEN queued_overload_time > execution_time THEN 'concurrency: queueing'
         WHEN bytes_spilled_to_remote_storage > 0   THEN 'memory: remote spill'
         WHEN bytes_spilled_to_local_storage > 0    THEN 'memory: local spill'
         WHEN partitions_scanned > partitions_total * 0.5 THEN 'io: poor pruning'
         ELSE 'cpu: operator cost'
       END AS likely_cause
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND execution_time > 60000
ORDER BY execution_time DESC
LIMIT 50;
```

Tuning is `QUERY_HISTORY` to find and classify, the query profile to confirm the operator, one change, then re-run and compare the same columns.

There are two sources with different freshness. `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` holds a year of history but can lag by up to about 45 minutes. The `QUERY_HISTORY` table function in `INFORMATION_SCHEMA` is much closer to real time and is the right source while you are still iterating on a query. Use the account view for trends and attribution, and the table function for the statement you just ran.

Filter before you aggregate. A sweep over every query in a day reads a lot of rows and buries the signal; start from the workload's warehouse and a narrow time window, exclude internal statements, and only then sort by `EXECUTION_TIME` or by spill. Operator-level attribution exists only in the profile of the single `query_id`.

## How do you diagnose and fix spilling?

```sql
-- Find the biggest spillers in the last day.
SELECT query_id, warehouse_name, warehouse_size,
       total_elapsed_time / 1000 AS elapsed_s,
       ROUND(bytes_spilled_to_local_storage  / POWER(1024, 3), 2) AS gb_local,
       ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_remote
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND bytes_spilled_to_local_storage + bytes_spilled_to_remote_storage > 0
ORDER BY bytes_spilled_to_remote_storage DESC, bytes_spilled_to_local_storage DESC
LIMIT 25;

-- Aggregate spill by warehouse to see whether it is systematic.
SELECT warehouse_name,
       COUNT(*) AS spilling_queries,
       ROUND(SUM(bytes_spilled_to_local_storage)  / POWER(1024, 4), 2) AS tb_local,
       ROUND(SUM(bytes_spilled_to_remote_storage) / POWER(1024, 4), 2) AS tb_remote
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
HAVING spilling_queries > 0
ORDER BY tb_remote DESC, tb_local DESC;
```

```sql
-- Before: a join on a non-unique key plus a late filter and an unnecessary sort.
SELECT o.customer_id, SUM(o.amount) AS revenue
FROM analytics.orders o
JOIN analytics.events e ON e.customer_id = o.customer_id
WHERE o.order_date >= '2026-01-01'
GROUP BY o.customer_id
ORDER BY revenue DESC;

-- After: filter and aggregate before the join, drop the sort.
WITH recent_orders AS (
  SELECT customer_id, SUM(amount) AS revenue
  FROM analytics.orders
  WHERE order_date >= '2026-01-01'
  GROUP BY customer_id
),
distinct_customers AS (
  SELECT DISTINCT customer_id FROM analytics.events
)
SELECT r.customer_id, r.revenue
FROM recent_orders r
JOIN distinct_customers c ON c.customer_id = r.customer_id;

-- If spill persists, give the operation more memory.
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';
```

Spilling means an operation outgrew warehouse memory — local spill is a slowdown, remote spill a severe one — so reduce what is held in memory first and raise the warehouse size second.

Spilling is reported per operator and summed per query. Local spill writes to the node's SSD; remote spill goes to object storage over the network and is an order of magnitude slower. Because a query can have several spilling operators, the query-level total can hide which one dominates; the profile shows it per node.

Any operator that builds a hash table or sorts a full input can spill: `JOIN`, `GROUP BY`, `ORDER BY`, `DISTINCT`, and window functions. Reducing rows before those operators is more effective than adding memory, because it shrinks the structure that must fit and also reduces the work every downstream operator does.

A bigger warehouse can hide a structural problem until the data grows. If a query spills again after one size increase, stop resizing and reshape the query, or the next data volume will spill on the larger warehouse too.

## Which query patterns make Snowflake slow?

```sql
-- 1. Function on the filter column: loses pruning.
SELECT COUNT(*) FROM analytics.orders WHERE YEAR(order_date) = 2026;
-- Rewrite as a raw range comparison.
SELECT COUNT(*) FROM analytics.orders
WHERE order_date >= '2026-01-01' AND order_date < '2027-01-01';

-- 2. Column list instead of SELECT *: less data through every operator.
SELECT * FROM analytics.orders WHERE order_date = '2026-10-04';
SELECT order_id, customer_id, amount FROM analytics.orders WHERE order_date = '2026-10-04';

-- 3. UNION deduplicates and sorts; UNION ALL does not.
SELECT customer_id FROM analytics.orders_2025
UNION
SELECT customer_id FROM analytics.orders_2026;
SELECT customer_id FROM analytics.orders_2025
UNION ALL
SELECT customer_id FROM analytics.orders_2026;

-- 4. ORDER BY without LIMIT sorts the whole result; add a bound when you only need the top rows.
SELECT customer_id, SUM(amount) AS revenue FROM analytics.orders
GROUP BY customer_id ORDER BY revenue DESC;
SELECT customer_id, SUM(amount) AS revenue FROM analytics.orders
GROUP BY customer_id ORDER BY revenue DESC LIMIT 100;

-- 5. Implicit VARCHAR-to-number join prevents a clean join plan.
SELECT * FROM analytics.orders o JOIN analytics.customers c ON o.customer_id = c.customer_id_str;
-- Cast explicitly once, before the join.
SELECT * FROM analytics.orders o
JOIN (SELECT TO_NUMBER(customer_id_str) AS customer_id FROM analytics.customers) c
  ON o.customer_id = c.customer_id;

-- 6. Distinct where the grain is already unique.
SELECT DISTINCT order_id, customer_id FROM analytics.orders;
SELECT order_id, customer_id FROM analytics.orders;

-- 7. Repeated VARIANT extraction: promote the field once into a typed view.
SELECT payload:user.id::NUMBER, COUNT(*) FROM raw.events GROUP BY 1;
CREATE OR REPLACE VIEW analytics.v_events AS
  SELECT payload:user.id::NUMBER AS user_id FROM raw.events;
SELECT user_id, COUNT(*) FROM analytics.v_events GROUP BY 1;
```

Avoid functions on filter columns, `SELECT *`, needless `DISTINCT` and `ORDER BY`, implicit type conversions, exploding joins, and repeated wide extraction — every one of them defeats an optimization Snowflake would otherwise apply for free.

The optimizer pushes filters to the scan and chooses join order from statistics. Anything that hides a literal or a type from it removes information it needs: a function, a cast, an implicit conversion, a `LIKE` with a leading wildcard, or a `DISTINCT` where the grain is already unique. Rewriting for the optimizer usually means making the predicate more literal and the plan more obvious, not making the query shorter.

Repeated work is a pattern too. Recomputing a stable aggregate on every dashboard refresh, re-scanning a table that changed in one partition, or extracting the same JSON path in every query spends credits on results that already exist. Before rewriting, ask whether the answer could be cached, materialized, or read from a clustered or optimized access path.

## How do you load data fast?

```sql
-- Compact and load a bounded slice, tolerating bad rows, cleaning up after success.
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Dry run before committing to a new format or feed.
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET)
  VALIDATION_MODE = 'RETURN_ERRORS';

-- Unload into many appropriately sized files instead of one giant file.
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue)
FILE_FORMAT = (TYPE = PARQUET)
MAX_FILE_SIZE = 100000000;

-- Load throughput and failures, by table and day.
SELECT DATE_TRUNC('day', last_load_time) AS load_day,
       table_name,
       COUNT(*) AS files,
       SUM(row_count) AS rows_loaded,
       SUM(error_count) AS errors
FROM snowflake.account_usage.copy_history
WHERE last_load_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY load_day DESC, rows_loaded DESC;

-- Any file that failed and why.
SELECT file_name, status, first_error_message, last_load_time
FROM snowflake.account_usage.copy_history
WHERE status = 'LOAD_FAILED'
  AND last_load_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY last_load_time DESC;
```

Load speed is file shape: aim for 100 to 250 MB compressed files, batch with `PATTERN`, use `VALIDATION_MODE` and a deliberate `ON_ERROR`, choose Snowpipe for latency and batch for cost, and watch `COPY_HISTORY`.

Load throughput is bounded by the slowest of three things: how many files can be processed in parallel, how fast each file can be read, and how fast the warehouse can write the resulting partitions. Very small files hurt the first and second, and a single huge file hurts the second, which is why `MAX_FILE_SIZE` on unload and file compaction before load attack the same bottleneck.

The two ingestion paths bill differently. `COPY INTO` runs on a warehouse you manage, so you control its size and lifecycle, and loading in large scheduled batches lets it suspend between runs. Snowpipe and Snowpipe Streaming run on Snowflake-managed serverless compute billed as credits attributed to the pipe or the streaming client, which is convenient for continuous arrival but less predictable per byte.

## How do you tune without burning credits?

```sql
-- Baseline and after, attributed to a query tag.
ALTER SESSION SET QUERY_TAG = 'tune:pruning_2026_10';

SELECT warehouse_name,
       DATE_TRUNC('hour', start_time) AS hour,
       SUM(credits_used_compute) AS compute_credits,
       SUM(credits_used_cloud_services) AS cloud_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY hour DESC;

-- Credits per query and per tag: the number that decides whether tuning paid.
SELECT query_tag,
       COUNT(*) AS queries,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) / NULLIF(COUNT(*), 0), 6) AS credits_per_query
FROM snowflake.account_usage.query_attribution_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY query_tag
ORDER BY credits DESC;

-- Guardrails while experimenting.
CREATE OR REPLACE RESOURCE MONITOR tune_monitor WITH CREDIT_QUOTA = 50
  FREQUENCY = DAILY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER WAREHOUSE tune_wh SET RESOURCE_MONITOR = tune_monitor
  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
```

```text
Tuning checklist
1. Is the query queued rather than slow? Check QUEUED_OVERLOAD_TIME.
2. Is it pruning? Compare PARTITIONS_SCANNED with PARTITIONS_TOTAL.
3. Is it spilled? Local spill is slow, remote spill is severe.
4. Are functions or casts on the filter column?
5. Is SELECT * pulling columns nothing uses?
6. Is the warehouse actually suspending between runs?
7. Did credits per query improve, not just latency?
8. Can the result be cached or precomputed instead of re-run?
```

Measure with `WAREHOUSE_METERING_HISTORY` and `QUERY_ATTRIBUTION_HISTORY`, tag the run, guard it with a resource monitor, and judge every change by credits per query — not by latency alone.

Attribution is what makes a tuning session provable. Setting a `QUERY_TAG` before the run lets `QUERY_ATTRIBUTION_HISTORY` group the before and the after, and `WAREHOUSE_METERING_HISTORY` shows the warehouse-level effect. Without a tag, the comparison is a guess against everything else that ran in the same hour.

Do the free checks first: was the query queued, did it prune, did it spill, did the warehouse actually suspend between runs. Only after those come changes that cost something to test, such as a clustering key or a resize, and those should run against a resource monitor with a low daily quota.

## Tuning exercises

These use only the standard library and model tuning arithmetic in the units the platform reports. Treat each assert as a check on the formula.

### Try it 1. Is this a pruning problem?

**Task.** Given the metrics for one query, compute the scanned-partition ratio and decide whether the query has a pruning problem. A query is considered adequately pruned when it scans at most half of the partitions.

**Hint.** `ratio = partitions_scanned / partitions_total`; a ratio above 0.5 means pruning is poor.

**Expected result.** `0.960 (96.0%)` scanned and the verdict `fix pruning`.

In [ ]:
metrics = {"partitions_scanned": 4800, "partitions_total": 5000}

ratio = metrics["partitions_scanned"] / metrics["partitions_total"]
verdict = "fix pruning" if ratio > 0.5 else "pruning is acceptable"

print(f"scanned ratio = {ratio:.3f} ({ratio * 100:.1f}%)")
print("verdict:", verdict)

assert round(ratio, 3) == 0.96
assert verdict == "fix pruning"

### Try it 2. Scale up or scale out?

**Task.** Using a credits-per-hour table, compute the credits consumed by a 600-second query on a MEDIUM warehouse and by the same query taking 300 seconds on a LARGE warehouse. Then compute the cost of running three SMALL clusters for a full hour and compare it with one MEDIUM cluster for a full hour.

**Hint.** `credits = credits_per_hour * seconds / 3600`. Doubling the size should roughly halve the runtime at about the same credits.

**Expected result.** The MEDIUM/600s and LARGE/300s runs each cost about `0.667` credits, so scale-up is credit-neutral. Three SMALL clusters cost `6.0` credits per hour versus `4.0` for one MEDIUM.

In [ ]:
CREDITS_PER_HOUR = {"XSMALL": 1, "SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


medium_run = credits("MEDIUM", 600)
large_run = credits("LARGE", 300)
print(f"MEDIUM 600s = {medium_run:.3f} credits")
print(f"LARGE  300s = {large_run:.3f} credits")
assert abs(medium_run - large_run) < 1e-9

scale_out = 3 * credits("SMALL", 3600)
scale_up = 1 * credits("MEDIUM", 3600)
print(f"3 x SMALL for 1h = {scale_out:.1f} credits")
print(f"1 x MEDIUM for 1h = {scale_up:.1f} credits")
assert scale_out == 6.0 and scale_up == 4.0

### Try it 3. Detect remote spill

**Task.** Write a function that classifies a metrics dictionary as `no spill`, `local spill`, or `remote spill`, then classify three example queries. Remote spill is the severe case because it means local disk was exhausted too.

**Hint.** Check `bytes_spilled_to_remote_storage` first; it takes precedence over local spill. Use `.get(key, 0)` so missing columns behave as zero.

**Expected result.** The three examples classify as `local spill`, `remote spill`, and `no spill`.

In [ ]:
def classify_spill(metrics):
    local = metrics.get("bytes_spilled_to_local_storage", 0)
    remote = metrics.get("bytes_spilled_to_remote_storage", 0)
    if remote > 0:
        return "remote spill"
    if local > 0:
        return "local spill"
    return "no spill"


local_only = {"bytes_spilled_to_local_storage": 2_000_000_000,
              "bytes_spilled_to_remote_storage": 0}
remote_too = {"bytes_spilled_to_local_storage": 1_000_000_000,
              "bytes_spilled_to_remote_storage": 500_000_000}
clean = {"bytes_spilled_to_local_storage": 0, "bytes_spilled_to_remote_storage": 0}

for name, m in [("query_a", local_only), ("query_b", remote_too), ("query_c", clean)]:
    print(name, "->", classify_spill(m))

assert classify_spill(local_only) == "local spill"
assert classify_spill(remote_too) == "remote spill"
assert classify_spill(clean) == "no spill"

### Try it 4. Choose a clustering key

**Task.** Given the approximate cardinality of four columns, pick a two-column clustering key that leads with the date used by range filters and then uses the highest-cardinality filtered column.

**Hint.** Keep the range column first, then take the column with the largest distinct count among the remaining candidates. `max(iterable, key=...)` selects it in one pass.

**Expected result.** The chosen key is `('order_date', 'customer_id')`.

In [ ]:
cardinality = {
    "order_date": 90,          # low cardinality, but range filters use it
    "customer_id": 2_000_000,  # high cardinality, also filtered
    "country": 40,             # nobody filters on it
    "event_type": 12,
}
range_column = "order_date"

id_column = max((c for c in cardinality if c != range_column), key=cardinality.get)
cluster_key = (range_column, id_column)
print("cluster key:", cluster_key)
print(f"{id_column} has {cardinality[id_column]:,} distinct values")

assert cluster_key == ("order_date", "customer_id")

### Try it 5. Multi-cluster scaling: STANDARD versus ECONOMY

**Task.** Simulate twelve ticks of arriving queries on a multi-cluster warehouse with at most three clusters. STANDARD adds a cluster as soon as queries queue; ECONOMY adds one only after three consecutive ticks with a non-empty queue. Compare total queue length with total cluster-ticks.

**Hint.** Each tick: finish work, add arrivals, apply the scaling policy, then start as many queries as there is spare capacity. Sum the queue length and the cluster count at the end of each tick.

**Expected result.** STANDARD keeps the queue at zero but pays 35 cluster-ticks; ECONOMY pays 29 cluster-ticks and leaves a queue of 14 query-ticks.

In [ ]:
def simulate(policy, ticks=12, arrivals_per_tick=1, runtime=3, max_clusters=3):
    clusters = 1
    active = []          # remaining ticks for each running query
    queue = 0
    queue_ticks = 0
    cluster_ticks = 0
    patient = 0          # consecutive ticks with a non-empty queue (ECONOMY only)
    for _ in range(ticks):
        active = [r - 1 for r in active if r - 1 > 0]
        queue += arrivals_per_tick
        if policy == "STANDARD":
            if queue > 0:
                clusters = min(max_clusters, clusters + 1)
        else:
            if queue > 0:
                patient += 1
                if patient % 3 == 0:
                    clusters = min(max_clusters, clusters + 1)
            else:
                patient = 0
        capacity = max(clusters - len(active), 0)
        started = min(capacity, queue)
        active += [runtime] * started
        queue -= started
        queue_ticks += queue
        cluster_ticks += clusters
    return queue_ticks, cluster_ticks


standard = simulate("STANDARD")
economy = simulate("ECONOMY")
print(f"STANDARD: queue_ticks={standard[0]:2d}  cluster_ticks={standard[1]}")
print(f"ECONOMY : queue_ticks={economy[0]:2d}  cluster_ticks={economy[1]}")

assert standard[0] < economy[0], "STANDARD should queue less"
assert standard[1] > economy[1], "STANDARD should pay for more clusters"

### Try it 6. Estimate cost per query

**Task.** Given a price of 3.00 currency units per credit, compute the credits and the cost of four queries and the average cost per query.

**Hint.** Cost is `credits * price`, and credits are `credits_per_hour * seconds / 3600`.

**Expected result.** Total cost is about `1.72`, an average of about `0.43` per query.

In [ ]:
PRICE_PER_CREDIT = 3.00
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}
queries = [("SMALL", 12), ("MEDIUM", 30), ("LARGE", 120), ("XLARGE", 60)]


def query_cost(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600 * PRICE_PER_CREDIT


costs = []
for size, seconds in queries:
    cost = query_cost(size, seconds)
    costs.append(cost)
    print(f"{size:<7} {seconds:>4}s -> {cost:.4f}")

total = sum(costs)
print(f"total = {total:.4f}, average = {total / len(costs):.4f}")
assert round(total, 2) == 1.72
assert round(total / len(costs), 2) == 0.43

### Try it 7. Read a cache-hit column

**Task.** Given the `percentage_scanned_from_cache` values for five runs of the same query, report the average and decide whether most runs were cache-served. Treat a run as cache-served when the value is above 90 percent.

**Hint.** Count the runs above the threshold and compare with half the number of runs.

**Expected result.** The average is `63.0` percent and most runs (3 of 5) were cache-served.

In [ ]:
scanned_from_cache = [98.5, 12.0, 99.1, 8.4, 97.0]

average = sum(scanned_from_cache) / len(scanned_from_cache)
cache_served = [value for value in scanned_from_cache if value > 90]
mostly_cached = len(cache_served) >= len(scanned_from_cache) / 2

print(f"average from cache = {average:.1f}%")
print(f"cache-served runs = {len(cache_served)} of {len(scanned_from_cache)}")
print("mostly cached:", mostly_cached)

assert round(average, 1) == 63.0
assert len(cache_served) == 3
assert mostly_cached is True

### Try it 8. Decide whether a scale-up experiment paid off

**Task.** A query took 60 seconds on a SMALL warehouse and 32 seconds on a MEDIUM warehouse. Compute the credits for both, the speedup, and decide whether the change is worth keeping. Treat it as worth keeping when the credits rise by no more than 10 percent and the speedup is at least 1.5x.

**Hint.** Compare credits, not size names, and remember that the smaller warehouse is not cheaper per credit-hour.

**Expected result.** Speedup is about `1.88x`, credits rise about `6.7` percent, so the scale-up is worth keeping.

In [ ]:
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


before = credits("SMALL", 60)
after = credits("MEDIUM", 32)
speedup = 60 / 32
credit_change = (after - before) / before
worth_keeping = credit_change <= 0.10 and speedup >= 1.5

print(f"SMALL 60s  = {before:.4f} credits")
print(f"MEDIUM 32s = {after:.4f} credits")
print(f"speedup = {speedup:.2f}x, credit change = {credit_change * 100:+.1f}%")
print("worth keeping:", worth_keeping)

assert round(speedup, 2) == 1.88
assert round(credit_change * 100, 1) == 6.7
assert worth_keeping is True

## Real world use cases for Snowflake

### 1. Retail ELT with dbt and incremental loads

A retailer lands point-of-sale, e-commerce, and inventory feeds in S3, loads them with `COPY INTO`, and transforms them with dbt models. Daily volume is roughly 400 million order lines and 2 TB of raw files, which makes file sizing, incremental models, and pruning the difference between a pipeline that finishes before the business day and one that does not.

```sql
COPY INTO raw.orders.orders FROM @raw.orders.landing/order_date=2026-10-04/
  FILE_FORMAT = (TYPE = PARQUET) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  PATTERN = '.*[.]parquet' ON_ERROR = 'CONTINUE' PURGE = TRUE;

MERGE INTO analytics.fct_orders AS t
USING stg_orders AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET amount = s.amount, status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (order_id, customer_id, order_date, amount, status, updated_at)
VALUES (s.order_id, s.customer_id, s.order_date, s.amount, s.status, s.updated_at);

ALTER TABLE analytics.fct_orders CLUSTER BY (order_date, customer_id);
```

Why: loading and transformation use separate warehouses, the merge makes retries safe, and clustering on `(order_date, customer_id)` keeps month-over-month reporting scans small.

### 2. Fintech regulatory reporting with Time Travel

A licensed lender must reproduce balances and classifications exactly as they stood on a reporting date. Loads are append-only and immutable, and every reporting query pins a timestamp so the number in the report can be regenerated months later by anyone with the same privileges.

```sql
CREATE TABLE analytics.regulatory_snapshot CLONE analytics.fct_balances
  AT (TIMESTAMP => '2026-09-30 23:59:59'::TIMESTAMP);

SELECT customer_id,
       SUM(CASE WHEN balance >= 10000 THEN balance ELSE 0 END) AS reportable_balance
FROM analytics.regulatory_snapshot
GROUP BY customer_id;

ALTER TABLE analytics.fct_balances SET DATA_RETENTION_TIME_IN_DAYS = 90;
```

Why: Time Travel plus clones make point-in-time reporting a query rather than a restore exercise, and secure views with masking policies keep PII out of the analyst role.

### 3. Multi-tenant SaaS analytics with row access policies

A SaaS vendor serves every customer from one copy of the data but must guarantee tenant isolation. A mapping table links each role to the tenant ids it may see, and a row access policy on the fact tables enforces the filter for every consumer, including BI tools that never add a tenant predicate of their own.

```sql
CREATE OR REPLACE ROW ACCESS POLICY security.tenant_policy
  AS (tenant_id STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() IN ('ACCOUNTADMIN', 'TENANT_ADMIN')
    OR EXISTS (
      SELECT 1 FROM security.tenant_map m
      WHERE m.role_name = CURRENT_ROLE() AND m.tenant_id = tenant_id
    );

ALTER TABLE analytics.fct_usage ADD ROW ACCESS POLICY security.tenant_policy ON (tenant_id);
```

Why: isolation is enforced by the engine rather than by each dashboard, so a misconfigured report cannot leak another tenant's data and one copy serves everyone.

### 4. Streaming CDC with Snowpipe Streaming and streams

A payments platform emits change events to Kafka, a connector writes them through Snowpipe Streaming into a raw table, a stream captures the delta, and a task merges it into the serving table every few minutes. End-to-end latency stays under ten minutes without any file landing step, which matters because payment status is customer-visible.

```sql
CREATE OR REPLACE TASK analytics.apply_payments
  WAREHOUSE = transform_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.payments.payments_stream')
AS
  MERGE INTO analytics.fct_payments AS t
  USING (
    SELECT payment_id, status, amount, updated_at, METADATA$ACTION AS action
    FROM raw.payments.payments_stream
    QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY updated_at DESC) = 1
  ) AS s
  ON t.payment_id = s.payment_id
  WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
  WHEN MATCHED THEN UPDATE SET status = s.status, amount = s.amount
  WHEN NOT MATCHED AND s.action = 'INSERT'
    THEN INSERT (payment_id, status, amount) VALUES (s.payment_id, s.status, s.amount);

ALTER TASK analytics.apply_payments RESUME;
```

Why: Snowpipe Streaming removes file-landing latency, the stream makes each task incremental, and the merge makes a retried task safe.

### 5. Cost-optimization program with resource monitors

A platform team runs dozens of warehouses and must cut spend without hurting SLAs. The program starts with attribution: credits per warehouse, per user, and per query tag, so the conversation moves from opinions to numbers. Then it right-sizes the biggest consumers, shortens auto-suspend on batch warehouses, and removes full refreshes that a `MERGE` or a dynamic table would do incrementally.

```sql
ALTER SESSION SET QUERY_TAG = 'team:marketing_pipeline';

SELECT warehouse_name,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) * 3.00, 2) AS approx_usd
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;

CREATE OR REPLACE RESOURCE MONITOR marketing_monitor WITH CREDIT_QUOTA = 500
  FREQUENCY = MONTHLY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
```

Why: credits are attributable per warehouse, user, and query tag, so the program is measured rather than argued about; typical wins are shorter auto-suspend, right-sizing, fewer full refreshes, and separated workload compute.

### 6. Lakehouse with Iceberg tables shared across engines

A media company keeps one copy of its event data as Iceberg on S3. Snowflake writes the curated tables, Spark reads and backfills them, and Trino serves ad-hoc queries against the same Parquet files and metadata. No engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer.

```sql
CREATE ICEBERG TABLE analytics.events (
  event_id STRING, event_date DATE, customer_id NUMBER, payload VARIANT)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'iceberg_vol'
  BASE_LOCATION = 'analytics/events';

INSERT INTO analytics.events
SELECT event_id, event_date, customer_id, payload
FROM raw.events_staging
WHERE event_date = '2026-10-04';
```

Why: the table is open, so no engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer, at the cost of making write ownership, maintenance, and catalog choice explicit.